# TP — Régression symbolique et méthode expérimentale

**Durée** : 2 h · **Prérequis** : Python de base · **Rien à installer**

### Ce que vous allez faire

1. Retrouver une loi physique à partir de données brutes
2. Mesurer l'effet du budget de calcul sur la qualité du résultat
3. **Tester si un mécanisme du moteur sert réellement à quelque chose**
4. Contraindre la recherche par l'analyse dimensionnelle

La partie 3 est le cœur du TP. Elle ne porte pas sur la régression
symbolique mais sur une compétence plus générale : **savoir si un résultat
que l'on observe est réel ou dû au hasard**.

---


## 0. Mise en route


In [ ]:
!pip install -q gp-elite
import numpy as np, time
import gp_elite, gp_elite.core as core
from gp_elite import symbolic_regression
print('GP_ELITE', gp_elite.__version__)


## 1. Retrouver une loi

On se donne 200 mesures d'une grandeur qui dépend d'une seule variable.
La loi qui les a produites vous est cachée pour l'instant.


In [ ]:
def jeu(n=200, seed=0):
    rng = np.random.RandomState(seed)
    x = rng.uniform(-3, 3, n)
    return np.column_stack([x]), x * np.sin(x**2) * np.cos(x/2)

X, y = jeu()
res = symbolic_regression(X, y, feature_names=['x'], operators='full',
                          generations=40, speed='fast', seed=0)
print(res.expression)


**Question 1.** La vraie loi est `x · sin(x²) · cos(x/2)`.
L'expression trouvée lui ressemble-t-elle ? Si non, comment expliquez-vous
qu'elle ajuste tout de même correctement les données ?


## 2. L'effet du budget

Un algorithme évolutionnaire explore un espace : plus il a de générations,
plus il explore. Mesurons.


In [ ]:
for g in (10, 30, 60):
    t = time.time()
    r = symbolic_regression(X, y, feature_names=['x'], operators='full',
                            generations=g, speed='fast', seed=0)
    mse = np.mean((y - r.predict(X))**2)
    print(f'{g:>3} générations : MSE = {mse:.3e}   taille = {r.size:<3} '
          f'({time.time()-t:.0f} s)')


**Question 2.** Le MSE diminue, mais la taille du modèle augmente.
Que se passerait-il avec 500 générations ? Quel risque voyez-vous ?


---

## 3. Un mécanisme sert-il vraiment ? (le cœur du TP)

GP_ELITE contient une **mémoire stigmergique** : une bibliothèque de
fragments d'expressions, pondérée par des phéromones, inspirée des colonies
de fourmis. Les fragments qui contribuent aux bons individus sont renforcés
et réutilisés.

L'idée est séduisante. Mais **est-ce qu'elle marche ?**

Ces mécanismes ne sont pas exposés par l'API publique. On les atteint en
interceptant la fonction qui construit la configuration du moteur.


In [ ]:
_original = core.make_cfg_nd
_drapeaux = {}

def _intercepte(*args, **kwargs):
    kwargs.update(_drapeaux)
    return _original(*args, **kwargs)

core.make_cfg_nd = _intercepte

SANS = dict(use_lib=False, use_cograph=False, use_seqmem=False)
AVEC = dict(use_lib=True,  use_cograph=True,  use_seqmem=True)

def essai(drapeaux, seed, generations=30):
    _drapeaux.clear(); _drapeaux.update(drapeaux)
    Xs, ys = jeu(seed=seed)
    r = symbolic_regression(Xs, ys, feature_names=['x'], operators='full',
                            generations=generations, speed='fast', seed=seed)
    return float(np.mean((ys - r.predict(Xs))**2))

print('outillage prêt')


### 3a. Une seule mesure


In [ ]:
a = essai(SANS, seed=0)
b = essai(AVEC, seed=0)
print(f'sans stigmergie : {a:.3e}')
print(f'avec            : {b:.3e}')
print(f'rapport         : x{b/a:.2f}')


**Question 3.** Au vu de ce seul résultat, que concluriez-vous ?
Écrivez votre conclusion **avant** de continuer.


### 3b. Cinq mesures

Le même essai, sur cinq jeux de données différents.
Comptez une à deux minutes.


In [ ]:
import math

sans, avec = [], []
for s in range(5):
    sans.append(essai(SANS, seed=s))
    avec.append(essai(AVEC, seed=s))
    print(f'seed {s} : sans={sans[-1]:.3e}  avec={avec[-1]:.3e}  '
          f"{'AVEC gagne' if avec[-1] < sans[-1] else 'SANS gagne'}")

victoires = sum(1 for a, b in zip(sans, avec) if b < a)
print(f'\nla stigmergie gagne sur {victoires} jeux sur 5')


### 3c. Est-ce du hasard ?

Si le mécanisme n'avait aucun effet, chaque jeu se déciderait à pile ou face.
La probabilité d'observer un écart au moins aussi marqué est le **test des
signes**. C'est cinq lignes de code.


In [ ]:
from math import comb

def test_des_signes(victoires, n):
    """Probabilité d'un écart au moins aussi marqué si l'effet était nul."""
    k = min(victoires, n - victoires)
    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2**n)

p = test_des_signes(victoires, 5)
print(f'p = {p:.3f}')
print('significatif' if p < 0.05 else "on ne peut rien conclure")


**Question 4.** Avec cinq jeux, quelle est la plus petite valeur de p
atteignable ? Combien de jeux faudrait-il au minimum pour espérer descendre
sous 0,05 ?

**Question 5.** Votre conclusion de la question 3 tient-elle toujours ?

> *Note.* L'auteur de GP_ELITE a mené cette mesure sur 4 problèmes et
> 15 jeux, soit 60 comparaisons appariées. Résultat : effet non démontrable,
> p ≈ 0,24. Le mécanisme est resté dans le code, documenté comme une
> caractéristique et non comme un avantage de performance.


---

## 4. Contraindre par la physique

Jusqu'ici le moteur cherche n'importe quelle formule. Or une loi physique
doit être **dimensionnellement homogène** : on n'additionne pas des mètres
et des kilogrammes.

Si l'on déclare les unités des colonnes, la recherche s'y restreint — et
peut même déduire une constante absente des données.


In [ ]:
from gp_elite import GPEliteRegressor

rng = np.random.RandomState(0)
allongement = rng.uniform(0.01, 0.10, 150).reshape(-1, 1)   # m
force = 250.0 * allongement[:, 0]                            # N

est = GPEliteRegressor(units=['m'], target_units='N',
                       unknown_constant=True,
                       generations=25, speed='fast', random_state=0)
est.fit(allongement, force)
print('unités de la constante :', est.constant_units_string())
print('valeur                 :', round(est.constant_value_, 4))


**Question 6.** La raideur d'un ressort s'exprime en N/m.
Vérifiez que `[kg / s^2]` en est bien l'écriture en unités de base.

**Question 7.** Pourquoi ce problème serait-il **impossible** si la
constante était forcée à être sans dimension ?


---

## À retenir

- Une formule qui ajuste bien les données n'est pas nécessairement la loi
- Un résultat sur un seul jeu de données ne conclut rien
- Un test statistique élémentaire suffit souvent à trancher
- Une connaissance du domaine — ici les unités — vaut mieux que du calcul

### Pour aller plus loin

- Le code du moteur est du Python lisible :
  [github.com/ariel95500-create/gp-elite](https://github.com/ariel95500-create/gp-elite)
- Les scripts de mesure du dépôt, dans `benchmarks/`, appliquent la même
  méthode que la partie 3, à plus grande échelle
- Une question, un bug, un échec sur vos données :
  [ouvrez une issue](https://github.com/ariel95500-create/gp-elite/issues/new/choose)
